# HW2 - Linear Regression from Scratch (Auto MPG)

**Dohyeong Ryu**

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
columns = ['mpg', 'cylinders', 'displacement', 'horsepower',
           'weight', 'acceleration', 'model_year', 'origin', 'car_name']

df = pd.read_csv('auto-mpg.data', sep=r'\s+', names=columns, na_values='?')

df = df.drop(columns=['car_name', 'origin'])

print('결측값 제거 전:', df.shape, '| 결측 개수:', int(df.isna().sum().sum()))
df = df.dropna().reset_index(drop=True)
print('결측값 제거 후:', df.shape)

features = ['cylinders', 'displacement', 'horsepower',
            'weight', 'acceleration', 'model_year']
X = df[features].values          
y = df['mpg'].values            
df.head()

결측값 제거 전: (398, 7) | 결측 개수: 6
결측값 제거 후: (392, 7)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
0,18.0,8,307.0,130.0,3504.0,12.0,70
1,15.0,8,350.0,165.0,3693.0,11.5,70
2,18.0,8,318.0,150.0,3436.0,11.0,70
3,16.0,8,304.0,150.0,3433.0,12.0,70
4,17.0,8,302.0,140.0,3449.0,10.5,70


In [8]:
def fit_linear_regression(X, y):
    X_b = np.c_[np.ones(len(X)), X]       
    beta = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
    intercept = beta[0]
    coefficients = beta[1:]
    return intercept, coefficients

def predict(X, intercept, coefficients):
    return X @ coefficients + intercept

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [12]:
n_iterations = 5
records = []

for i in range(n_iterations):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i)

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)

    intercept, coefs = fit_linear_regression(X_train_s, y_train)

    y_pred = predict(X_test_s, intercept, coefs)
    err = rmse(y_test, y_pred)

    records.append([intercept, *coefs, err])
    print(f'Exp.{i+1}  RMSE = {err:.4f}')

Exp.1  RMSE = 3.4465
Exp.2  RMSE = 3.5861
Exp.3  RMSE = 3.4774
Exp.4  RMSE = 3.3849
Exp.5  RMSE = 3.4035


In [13]:
results = pd.DataFrame(
    records,
    columns=['intercept'] + features + ['RMSE'],
    index=[f'Exp. {i+1}' for i in range(n_iterations)]
)

results.loc['Mean'] = results.mean()
results.round(4)

,intercept,cylinders,displacement,horsepower,weight,acceleration,model_year,RMSE
Exp. 1,23.4824,-0.4172,0.7476,0.1510,-5.9301,0.4138,2.7015,3.4465
Exp. 2,23.4042,0.0885,0.5502,0.2756,-6.2357,0.4685,2.7459,3.5861
Exp. 3,23.6157,-0.5939,0.6418,-0.2897,-5.4128,0.1137,2.8448,3.4774
Exp. 4,23.6757,-0.7935,1.1757,-0.2893,-5.5151,0.2163,2.8969,3.3849
Exp. 5,23.7460,-0.2958,0.6609,0.1739,-6.0559,0.0406,2.7321,3.4035
Mean,23.5848,-0.4024,0.7552,0.0043,-5.8299,0.2506,2.7842,3.4597


In [14]:
print(f"평균 RMSE (5회): {results.loc['Mean', 'RMSE']:.4f} mpg")
print(f"RMSE 표준편차 : {pd.Series([r[-1] for r in records]).std():.4f}")

평균 RMSE (5회): 3.4597 mpg
RMSE 표준편차 : 0.0794
